In [1]:
%load_ext autoreload
%autoreload 2

In [5]:
import os
from dotenv import load_dotenv
_ = load_dotenv('./.env', override=True)

In [30]:
from azure_aad import get_or_refresh_token
from openai import AzureOpenAI, AsyncAzureOpenAI
from aoai_client import AOAI
from deepeval.metrics import ContextualRelevancyMetric
from deepeval.test_case import LLMTestCase
from src.llm.prompt_templates import create_context_blocks
from src.database.database_utils import get_weaviate_client
from src.evaluation.retrieval_evaluation import execute_evaluation
from src.evaluation.llm_evaluation import TestCaseGenerator, EvalResponse, CustomAzureOpenAI, load_eval_response
from src.preprocessor.preprocessing import FileIO
from src.reranker import ReRanker
from typing import Callable
from tqdm import tqdm
from tqdm.asyncio import tqdm_asyncio
from asyncio import Semaphore, sleep
import nest_asyncio
nest_asyncio.apply()

In [46]:
client = AOAI(model=deployment_name, asynch=True)

In [47]:
response = await client.aquery('How many days until Christmas, starting from today.  Today is May 27th, 2024.')

### Load Evaluation Constants

In [8]:
azure_model = CustomAzureOpenAI(deployment_name='gpt-4o')
emb_model_name = 'BAAI/bge-base-en'
retriever = get_weaviate_client(model_name_or_path=emb_model_name)
retriever.model_name_or_path

'BAAI/bge-base-en'

In [9]:
retriever.show_all_collections()

['Huberman_bgebase_256',
 'Huberman_minilm_256',
 'Huberman_minilm_258',
 'Huberman_distilbert_256',
 'Huberman_bge_base_finetuned_500']

In [10]:
collection_name = 'Huberman_bgebase_256'
golden_dataset = FileIO.load_json('data/golden_datasets/golden_256_hard.json')

In [11]:
reranker=ReRanker()

In [74]:
evaluation = execute_evaluation(golden_dataset,
                                collection_name=collection_name,
                                retriever=retriever, 
                                reranker=None,
                                retrieve_limit=10,
                                top_k=4,
                                search_type=['hybrid']
                               )

Queries: 100%|███████████████████████████████████████████████████| 100/100 [00:27<00:00,  3.70it/s]


Total Processing Time: 0.45 minutes

### Contextual Relevancy Metric

In [12]:
queries = [v for v in golden_dataset['queries'].values()]

In [15]:
model = ''
retriever = get_weaviate_client()
retriever.model_name_or_path

'sentence-transformers/all-MiniLM-L6-v2'

In [16]:
collection_name = 'Huberman_minilm_256'

### Create Test Cases

In [17]:
search_results = [retriever.hybrid_search(query, collection_name, limit=4) for query in tqdm(queries, 'QUERIES', position=0, leave=True)]

QUERIES: 100%|███████████████████████████████████████████████████████████████████████████| 100/100 [00:19<00:00,  5.09it/s]


In [18]:
contexts = [[f'Guest: {d["guest"]}\n{d["content"]}' for d in result] for result in search_results]

In [19]:
test_cases = [LLMTestCase(input=q, actual_output='', retrieval_context=c) for q,c in zip(queries, contexts)]

In [22]:
FileIO.save_as_json('./data/golden_datasets/context_relevancy_testcases_100.json',[t.__dict__ for t in test_cases])

2024-05-28 09:04:27.614 | INFO     | src.preprocessor.preprocessing:save_as_json:111 - Data saved as json file here: ./data/golden_datasets/context_relevancy_testcases_100.json


### Define Metric Functions

In [45]:
async def context_relevancy_async(test_case: LLMTestCase,
                                  model: str,
                                  sem: Semaphore,
                                  index: int
                                  ) -> EvalResponse:
    async with sem:
        metric = ContextualRelevancyMetric(model=model)
        await metric.a_measure(test_case, False)
        response = load_eval_response(metric, test_case)
        print(f'Response {index} completed.')
        if index < 50:
            await sleep(60)
        return response

In [46]:
async def async_evaluation(test_cases: list[LLMTestCase],
                           model: str,
                           async_metric_call: Callable,
                           rpm: int
                           ) -> list[EvalResponse]:
    sem = Semaphore(rpm)
    tasks = [async_metric_call(case, model, sem, i) for i, case in enumerate(test_cases)]
    responses = await tqdm_asyncio.gather(*tasks)
    return responses

In [172]:
#single test
# result = await context_relevancy_async(test_cases[0], azure_model)

In [47]:
results = await async_evaluation(test_cases, azure_model, context_relevancy_async, 50)


  0%|                                                                                           | 0/100 [00:00<?, ?it/s]/opt/anaconda3/envs/vsa/lib/python3.10/asyncio/tasks.py:94: ResourceWarning: unclosed <socket.socket fd=161, family=AddressFamily.AF_INET, type=SocketKind.SOCK_STREAM, proto=6, laddr=('192.168.4.117', 62171), raddr=('20.62.58.5', 443)>
  super().__init__(loop=loop)
/opt/anaconda3/envs/vsa/lib/python3.10/asyncio/selector_events.py:710: ResourceWarning: unclosed transport <_SelectorSocketTransport fd=111 read=idle write=<idle, bufsize=0>>
  _warn(f"unclosed transport {self!r}", ResourceWarning, source=self)
/opt/anaconda3/envs/vsa/lib/python3.10/asyncio/selector_events.py:710: ResourceWarning: unclosed transport <_SelectorSocketTransport fd=108 read=idle write=<idle, bufsize=0>>
  _warn(f"unclosed transport {self!r}", ResourceWarning, source=self)
/opt/anaconda3/envs/vsa/lib/python3.10/asyncio/selector_events.py:710: ResourceWarning: unclosed transport <_SelectorSocket

Response 73 completed.



  1%|▊                                                                               | 1/100 [03:18<5:26:43, 198.01s/it]

Response 87 completed.
Response 59 completed.
Response 61 completed.
Response 56 completed.
Response 97 completed.
Response 85 completed.
Response 58 completed.
Response 52 completed.
Response 1 completed.
Response 45 completed.
Response 41 completed.
Response 5 completed.
Response 83 completed.
Response 98 completed.
Response 34 completed.
Response 30 completed.
Response 90 completed.
Response 75 completed.
Response 91 completed.
Response 17 completed.
Response 81 completed.
Response 11 completed.
Response 47 completed.
Response 71 completed.
Response 19 completed.
Response 66 completed.
Response 51 completed.
Response 44 completed.
Response 37 completed.
Response 40 completed.
Response 16 completed.
Response 2 completed.
Response 69 completed.
Response 78 completed.
Response 54 completed.
Response 63 completed.
Response 39 completed.
Response 93 completed.
Response 53 completed.
Response 76 completed.
Response 4 completed.
Response 94 completed.
Response 13 completed.



  2%|█▌                                                                               | 2/100 [03:20<2:15:53, 83.20s/it]

...


 70%|█████████████████████████████████████████████████████████▍                        | 70/100 [06:44<00:29,  1.02it/s]

Response 70 completed.
Response 24 completed.
Response 57 completed.
Response 3 completed.
Response 28 completed.
Response 27 completed.



 72%|███████████████████████████████████████████████████████████                       | 72/100 [06:45<00:24,  1.15it/s]

Response 0 completed.
Response 80 completed.
Response 26 completed.
Response 36 completed.



100%|█████████████████████████████████████████████████████████████████████████████████| 100/100 [07:47<00:00,  4.68s/it]


In [48]:
len(results)

100

In [50]:
scores = [r.score for r in results]
sum(scores)/len(scores)

0.5225

In [58]:
mask = np.array(scores) <= 0

In [76]:
indices = [score for alist in np.argwhere(mask).tolist() for score in alist]
indices[-1]

91

In [75]:
for i in indices:
    print(results[i].reason)
    print('\n\n')

The score is 0.00 because "the context does not specifically address 'the ability to mentalize' and its impact on relationships" and focuses instead on "the benefits of a gratitude practice on social relationships and overall wellbeing."



The score is 0.00 because the context "focuses more on learning during sleep, non-sleep deep rest (NSDR), and spatial memory tasks," and does not address "the relationship between temperature and sleep quality," essential for the input question about memory retention and learning potential.



The score is 0.00 because the context "does not provide recent research findings or discussions on the implications of unconventional sleep schedules like the Uberman schedule on productivity and overall well-being" and "spends a significant portion discussing unrelated topics."



The score is 0.00 because the context does not address the importance of reevaluating and adapting our approach to life when faced with unexpected challenges or limitations; instead

In [79]:
import json

In [80]:
with open('/Users/americanthinker/Downloads/trec-covid/corpus.jsonl') as f:
    data = [json.loads(line) for line in f.readlines()]

In [81]:
len(data)

171332

In [83]:
data[100000]

{'_id': 'um3djluz',
 'title': 'A Survey on Applications of Artificial Intelligence in Fighting Against COVID-19',
 'text': 'The COVID-19 pandemic caused by the SARS-CoV-2 virus has spread rapidly worldwide, leading to a global outbreak. Most governments, enterprises, and scientific research institutions are participating in the COVID-19 struggle to curb the spread of the pandemic. As a powerful tool against COVID-19, artificial intelligence (AI) technologies are widely used in combating this pandemic. In this survey, we investigate the main scope and contributions of AI in combating COVID-19 from the aspects of disease detection and diagnosis, virology and pathogenesis, drug and vaccine development, and epidemic and transmission prediction. In addition, we summarize the available data and resources that can be used for AI-based COVID-19 research. Finally, the main challenges and potential directions of AI in fighting against COVID-19 are discussed. Currently, AI mainly focuses on medic